# Deploy Olist Freight Model with Python SDK

This notebook shows a Python (SDK) alternative to the existing CLI deployment.

It registers an MLflow model from local files (MLmodel, model.pkl, conda.yaml, requirements.txt), deploys it to a managed online endpoint, and invokes the endpoint.

Reference: https://learn.microsoft.com/azure/machine-learning/how-to-deploy-mlflow-models-online-endpoints?view=azureml-api-2

## Prerequisites

- Azure CLI logged in: `az login`
- Access to the Azure ML workspace (Contributor or similar)
- Python packages: `azure-ai-ml`, `azure-identity`, `requests`

This notebook defaults to the teaching workspace we used in class.

In [ ]:
# If needed, install dependencies for this notebook kernel
%pip -q install azure-ai-ml azure-identity requests

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import requests
from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import CodeConfiguration, Environment, ManagedOnlineDeployment, ManagedOnlineEndpoint, Model
from azure.identity import AzureCliCredential

In [ ]:
# Workspace settings
subscription_id = '42848f6c-f0f7-4143-b9d3-76441ceeda99'
resource_group = 'rg-ai300-le6066d546ae94e0d84'
workspace_name = 'mlw-ai300-le6066d546ae94e0d84'
location = 'eastus'

# Naming (adjust if needed; endpoint names must be unique within the region)
model_name = 'olist-freight-cost-py'
endpoint_name = 'freight-olist-kvsrd-py'
deployment_name = 'blue'

# Local paths
here = Path('.').resolve()
mlflow_model_dir = here / 'mlflow-model' / 'olist-freight-cost' / 'model'
deploy_dir = here / 'deploy'
sample_request_file = here / 'sample-request.json'

assert mlflow_model_dir.exists(), f'Missing MLflow model folder: {mlflow_model_dir}'
assert (mlflow_model_dir / 'MLmodel').exists(), 'MLmodel file not found in the model folder'
assert sample_request_file.exists(), f'Missing sample request: {sample_request_file}'

In [ ]:
# Connect to the Azure ML workspace using your Azure CLI login
ml_client = MLClient(AzureCliCredential(), subscription_id, resource_group, workspace_name)
ml_client

In [ ]:
# Register the MLflow model from local files
model = ml_client.models.create_or_update(
    Model(
        name=model_name,
        path=str(mlflow_model_dir),
        type=AssetTypes.MLFLOW_MODEL,
        description='Olist freight-cost model (registered via Python notebook from MLflow artifacts)',
    )
)
model

In [ ]:
# Create (or update) an online endpoint
endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description='Freight cost endpoint (deployed via Python SDK notebook)',
    auth_mode='key',
    tags={'model': model_name},
)
endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()
endpoint.name

In [ ]:
# Define the environment + scoring script (custom contract, same as the CLI deployment)
environment = Environment(
    name='olist-freight-inference-py',
    image='mcr.microsoft.com/azureml/minimal-ubuntu22.04-py39-cpu-inference:latest',
    conda_file=str(deploy_dir / 'conda.yaml'),
)
environment = ml_client.environments.create_or_update(environment)
environment

In [ ]:
# Deploy the model
deployment = ManagedOnlineDeployment(
    name=deployment_name,
    endpoint_name=endpoint_name,
    model=model,
    environment=environment,
    code_configuration=CodeConfiguration(code=str(deploy_dir), scoring_script='score.py'),
    instance_type='Standard_D2ds_v5',
    instance_count=1,
)
deployment = ml_client.online_deployments.begin_create_or_update(deployment).result()
deployment

In [ ]:
# Route 100% of traffic to this deployment
endpoint = ml_client.online_endpoints.get(endpoint_name)
endpoint.traffic = {deployment_name: 100}
endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print('Scoring URI:', endpoint.scoring_uri)

In [ ]:
# Fetch the key from Azure and invoke the endpoint
keys = ml_client.online_endpoints.get_keys(endpoint_name)
key = keys.primary_key

payload = json.loads(sample_request_file.read_text(encoding='utf-8'))
resp = requests.post(
    endpoint.scoring_uri,
    json=payload,
    headers={'Authorization': f'Bearer {key}'},
    timeout=30,
)
print('HTTP', resp.status_code)
print(resp.json())

In [ ]:
# Optional: invoke using the Azure ML SDK (request_file)
response = ml_client.online_endpoints.invoke(
    endpoint_name=endpoint_name,
    request_file=str(sample_request_file),
)
response

## Cleanup

Online endpoints bill while the deployment exists. Delete the endpoint when you are done.

In [ ]:
# Uncomment to delete the endpoint (stops cost)
# ml_client.online_endpoints.begin_delete(name=endpoint_name).result()